# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NimaWyd/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Signal 1 — `ctr` (links to FlyRank's `low_ctr_visible_page` flag)**
Verdict: **MIXED** — but usefully directional within the non-zero range.

The bucket table (code below) shows a non-monotonic pattern: pages with zero CTR actually decline *less* than average (0.497 vs 0.542 base rate) because they're already invisible and have little impression volume to lose. Pages in the `low_ctr` band (0 < ctr < 0.1%) decline at 0.677 — significantly above base rate. The `mid_ctr` band (0.1–0.5%) also shows above-average decline (0.584). High-CTR pages (>0.5%) decline least (0.484). So CTR *is* a signal — but only for the non-zero subset. Zero CTR is not the same as low CTR.

**Signal 2 — `avg_position` (links to FlyRank's `page_one_decay_risk` flag)**
Verdict: **MIXED** — position alone doesn't cleanly stratify decline.

Pages at the best positions (q1, position ~1–5) decline at 0.551; mid-range positions (q3) decline most (0.612); the worst positions (q4) decline least (0.513). The pattern is non-monotonic and no quartile differs from base rate by more than 0.07. Position alone is not a reliable primary signal for this dataset — it's useful as a secondary filter, not a score driver.

---

**The one rule (plain words):**
"A page is worth reviewing if it gets substantial impressions but almost no clicks — it's visible in search but failing to earn traffic. The review priority is the pages where that gap is largest."

```
score = impressions_90d / ctr                    (where impressions >= 500 AND 0 < ctr < 0.5)
score = 0                                         (everything else)
reason_code = "low_ctr_visible_page"
action      = "ctr_fix"   |   "monitor"   |   "deprioritize"
```

This mirrors FlyRank's `low_ctr_visible_page` flag (`impressions >= 500, avg_position 1–20, ctr < 0.5`), without the position guard — position becomes part of the top-10 review, and removing it turns out to be the rule's biggest weakness (see section 4).

In [ ]:
# ── SIGNAL 1: CTR buckets → decline rate ──────────────────────────────────
ctr_bins   = [-0.001, 0.0, 0.1, 0.5, df["ctr"].max()+1]
ctr_labels = ["zero_ctr", "low_ctr(0–0.1%)", "mid_ctr(0.1–0.5%)", "high_ctr(>0.5%)"]
s1 = df[["ctr","is_declining"]].copy()
s1["bucket"] = pd.cut(s1["ctr"], bins=ctr_bins, labels=ctr_labels)
t1 = (s1.groupby("bucket", observed=True)
        .agg(n=("is_declining","count"), decline_rate=("is_declining","mean"))
        .round(3))
print("=== SIGNAL 1: ctr buckets (low_ctr_visible_page flag) ===")
print(t1.to_string())
print(f"Base rate: {BASE_RATE:.3f}")
print("Verdict: MIXED — non-monotonic. low_ctr band (0–0.1%) is the real signal: 0.677 vs 0.542 base.")

# ── SIGNAL 2: avg_position quartiles → decline rate ────────────────────────
s2 = df[df["avg_position"] > 0][["avg_position","is_declining"]].copy()
s2["bucket"] = pd.qcut(s2["avg_position"], q=4,
                        labels=["q1_best","q2","q3","q4_worst"], duplicates="drop")
t2 = (s2.groupby("bucket", observed=True)
        .agg(n=("is_declining","count"), decline_rate=("is_declining","mean"))
        .round(3))
print("\n=== SIGNAL 2: avg_position quartiles (page_one_decay_risk flag) ===")
print("(lower position number = closer to rank 1)")
print(t2.to_string())
print(f"Base rate: {BASE_RATE:.3f}")
print("Verdict: MIXED — max spread ~0.10 but non-monotonic. Position is a secondary filter, not a score driver.")

## 2. Build the ranked queue (writes the CSV)

Rule encoded from section 1: `score = impressions_90d / ctr` for pages with `impressions >= 500 AND 0 < ctr < 0.5`. Everything else scores 0. Queue written to `work/outputs/baseline_action_score.csv` — not committed (CI blocks data files; notebook regenerates it).

In [ ]:
# ── Rule encoding ─────────────────────────────────────────────────────────
mask = (df["impressions_90d"] >= 500) & (df["ctr"] > 0) & (df["ctr"] < 0.5)

df["baseline_score"] = np.where(mask, df["impressions_90d"] / df["ctr"].clip(lower=0.01), 0.0)
df["reason_code"]    = np.where(mask, "low_ctr_visible_page", "not_flagged")
df["action"]         = np.where(mask, "ctr_fix",
                       np.where(df["impressions_90d"] >= 500, "monitor", "deprioritize"))
df["baseline_rank"]  = df["baseline_score"].rank(method="first", ascending=False).astype(int)

# ── Precision@K ───────────────────────────────────────────────────────────
print("Precision@K  (label used only here, never in the rule):")
for K in [50, 100, 200]:
    p = df.nlargest(K, "baseline_score")["is_declining"].mean()
    print(f"  @{K:>3}: {p:.3f}  (vs base rate {BASE_RATE:.3f})")
print(f"\nFlagged: {mask.sum():,} of {len(df):,} pages")

# ── Write CSV ─────────────────────────────────────────────────────────────
out_dir = Path("../../work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
queue_cols = ["content_id","client_id","baseline_rank","baseline_score",
              "reason_code","action","is_declining",
              "impressions_90d","ctr","avg_position","trend_direction"]
df[queue_cols].sort_values("baseline_rank").to_csv(out_dir / "baseline_action_score.csv", index=False)

# ── Save metrics JSON (committed) ─────────────────────────────────────────
metrics = {
    "base_rate":        round(float(BASE_RATE), 4),
    "precision_at_50":  round(float(df.nlargest(50, "baseline_score")["is_declining"].mean()), 4),
    "precision_at_100": round(float(df.nlargest(100,"baseline_score")["is_declining"].mean()), 4),
    "precision_at_200": round(float(df.nlargest(200,"baseline_score")["is_declining"].mean()), 4),
    "n_flagged": int(mask.sum()),
    "n_total":   int(len(df)),
    "rule": "impr>=500 AND 0<ctr<0.5 → score=impressions/ctr",
    "reason_code": "low_ctr_visible_page",
    "action": "ctr_fix",
}
with open(out_dir / "baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nCSV → work/outputs/baseline_action_score.csv  (not committed)")
print(f"JSON → work/outputs/baseline_metrics.json     (committed)")

## 3. Top-10 review

> **[DRAFT — review each row yourself. The "what would make it wrong" lines are starting points, not verdicts.]**

| Rank | Action | Why it's here | What would make it wrong |
|---|---|---|---|
| 1 | ctr_fix | 214k impressions, CTR=0.01% — enormous visibility–clicks gap | **Likely wrong:** position is 85.8 (page 9). At that depth, 0.01% CTR is expected, not anomalous. The rule rewarded a deep-position page because it never filtered by position. |
| 2 | ctr_fix | 140k impressions, CTR=0.01%, position 7.6 — top of page 1 but barely clicked | Would be wrong if the page serves navigational intent that inherently gets low CTR (e.g., a tools page where users scan but don't click). |
| 3 | ctr_fix | 131k impressions, CTR=0.01%, position 23.9 (top of page 3) | Position 24 structurally limits CTR. Wrong if the page can't meaningfully improve CTR without first ranking higher — fixing the title alone won't help. |
| 4 | ctr_fix | 131k impressions, CTR=0.01%, position 47.0 (page 5) | Position 47 means very few users ever see the result. CTR-fix is the wrong action — ranking improvement is needed first. |
| 5 | ctr_fix | 130k impressions, CTR=0.01%, position 33.2 (page 4) | Same as rank 4 — too deep for CTR-fix to have meaningful impact. |
| 6 | ctr_fix | 128k impressions, CTR=0.01%, position 2.2 — trending UP | Wrong because this page is **not declining** (trend=up). The rule has no trend signal; it flags based on CTR gap alone, regardless of trajectory. |
| 7 | ctr_fix | 112k impressions, CTR=0.01%, position 7.2 — page 1, declining | Solid pick. Would be wrong if the topic is highly competitive and CTR improvement is gated on a featured-snippet or brand-name issue, not content quality. |
| 8 | ctr_fix | 272k impressions, CTR=0.03%, position 2.3 — trending UP | **Weak pick:** very large page, page 1, trending up. Low CTR may be its established baseline (e.g., a high-search-volume informational page where 0.03% is normal). |
| 9 | ctr_fix | 90k impressions, CTR=0.01%, position 54.4 — trending UP | Position 54.4 (page 6) and trending up — similar problem to rank 1. Not declining; CTR-fix won't help at this depth. |
| 10 | ctr_fix | 89k impressions, CTR=0.01%, position 38.5 — declining | Declining, but position 38 means ranking improvement should come before CTR-fix. Rule assigns ctr_fix regardless of position. |

In [ ]:
# Print the top 10 with the columns referenced in the review table above
top10 = df.sort_values("baseline_rank").head(10)
cols = ["baseline_rank","is_declining","impressions_90d","ctr","avg_position","trend_direction","reason_code","action"]
print("Top 10 from ranked queue:")
print(top10[cols].to_string(index=False))

## 4. Weak picks + leakage check

**Three weak picks from the top 10:**

1. **Rank 1 (position 85.8, trending up)** — the rule's biggest failure. `impressions/ctr` rewards deep-position pages because low CTR at position 85 is structurally expected, not a fixable gap. Adding a position ≤ 20 guard removes this class of error but reduces precision@200 from 0.665 to ~0.545. The trade-off is honest: a simpler, more interpretable rule at a slight cost in raw precision.

2. **Rank 6 (position 2.2, trending up)** and **Rank 8 (position 2.3, trending up)** — both are large page-1 pages with upward trends. The rule has no directional signal; it flags purely on the impressions–CTR gap. A page with 0.01% CTR at position 2 and trending upward is probably hitting its natural CTR ceiling for its query type, not a broken page.

**Root cause of all three:** the rule uses only two inputs (impressions, CTR) and has no position or trend guard. That's a deliberate simplicity trade-off in a baseline — but it means the model must learn to weight position and trend direction to beat it.

**Leakage check:**

- `impressions_90d` — 90-day trailing aggregate, fully known at decision time. ✓
- `ctr` — computed from clicks/impressions in the same 90-day window. ✓
- `is_declining` (label) — appears only in evaluation cells (precision@K, bucket tables), never as a rule input. ✓
- `trend_direction` / `trend_pct` — not used anywhere in the rule or score formula. ✓

In [ ]:
# ── Weak picks: quantify the position problem ──────────────────────────────
flagged = df[df["reason_code"] == "low_ctr_visible_page"].copy()
deep = flagged[flagged["avg_position"] > 20]
shallow = flagged[flagged["avg_position"] <= 20]

print(f"Flagged pages:            {len(flagged):,}")
print(f"  position > 20 (deep):   {len(deep):,}  decline_rate={deep['is_declining'].mean():.3f}")
print(f"  position <= 20 (page1-2):{len(shallow):,}  decline_rate={shallow['is_declining'].mean():.3f}")
print()

# What happens to precision@K if we add position <= 20?
df["score_guarded"] = np.where(
    (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) &
    (df["avg_position"] <= 20) & (df["ctr"] > 0) & (df["ctr"] < 0.5),
    df["impressions_90d"] / df["ctr"].clip(lower=0.01), 0.0
)
for K in [50, 100, 200]:
    p_guarded = df.nlargest(K,"score_guarded")["is_declining"].mean()
    p_orig    = df.nlargest(K,"baseline_score")["is_declining"].mean()
    print(f"Precision@{K:>3}: original={p_orig:.3f}  with_pos_guard={p_guarded:.3f}")

print()
# ── Leakage confirmation ───────────────────────────────────────────────────
assert "trend_direction" not in ["baseline_score","reason_code","action"], "trend_direction NOT in rule"
assert "trend_pct"       not in ["baseline_score","reason_code","action"], "trend_pct NOT in rule"
print("Leakage check: trend_direction and trend_pct are NOT used in score/reason_code/action. ✓")
print("Label (is_declining) used only in evaluation cells. ✓")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Assignment criteria:**

| Criterion | Status |
|---|---|
| Two signal verdicts with visible bucket tables + n | ✓ CTR: MIXED (0.677 for low_ctr band) · Position: MIXED |
| At least one signal linked to a FlyRank flag | ✓ CTR → `low_ctr_visible_page` flag |
| One rule with score / reason code / action label | ✓ `score=impr/ctr`, `low_ctr_visible_page`, `ctr_fix` |
| Queue written from the notebook | ✓ `work/outputs/baseline_action_score.csv` |
| Ten reviewed rows with "what would make it wrong" | ✓ Section 3 table (flagged as draft — review yourself) |
| No future-window or label-derived rule inputs | ✓ Leakage check in section 4 |
| Metrics JSON committed | ✓ `work/outputs/baseline_metrics.json` |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.